In [1]:
print('hello world')

hello world


In [2]:
import gymnasium as gym
import numpy as np

env = gym.make("CartPole-v1")
print("观测上限:", env.observation_space.high)
print("观测下限:", env.observation_space.low)
print("小车位置阈值:", env.unwrapped.x_threshold)
print(
    "杆子角度阈值:",
    np.degrees(env.unwrapped.theta_threshold_radians),
    "度",
)

观测上限: [4.8               inf 0.41887903        inf]
观测下限: [-4.8               -inf -0.41887903        -inf]
小车位置阈值: 2.4
杆子角度阈值: 12.0 度


# PPO 训练 CartPole —— 逐块运行版

本 notebook 是 `1-ppo_cartpole_simple.py` 的拆分版本：把脚本按逻辑切成若干 block，
逐个运行、逐个看输出，就能理解每一步在做什么。
读完这里，再去看 `1-ppo_cartpole.py`，新增的就只有 SwanLab 日志那一层。

## 1. 导入依赖与选择设备

`device_utils` 在 `code/` 目录下，所以先把它加入 `sys.path`。
`resolve_sb3_device("auto")` 的优先级是 CUDA → Apple MPS → CPU。

In [1]:
import os
import sys
from pathlib import Path

# notebook 在 code/chapter01_cartpole/ 下，把上一级 code/ 加入 sys.path 才能 import device_utils
_CODE_ROOT = Path.cwd().parent if Path.cwd().name == "chapter01_cartpole" else Path.cwd()
if str(_CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(_CODE_ROOT))

import gymnasium as gym
import numpy as np
from stable_baselines3 import PPO
from stable_baselines3.common.evaluation import evaluate_policy

from device_utils import describe_device, resolve_sb3_device, print_device_report

os.makedirs("output", exist_ok=True)

device = resolve_sb3_device("auto")   # 也可以手动写 "cpu" / "mps" / "cuda"
print_device_report(device)

PyTorch device report
  PyTorch:  2.12.0
  CUDA:     False
  MPS:      True
  Selected: Apple MPS (Metal) (mps)


device(type='mps')

## 2. 创建环境，看看它长什么样

CartPole-v1：观测是 4 维连续向量（小车位置、速度、杆子角度、角速度），
动作是 2 个离散值（向左推 / 向右推）。
超出位置或角度阈值，回合就结束。

In [2]:
env = gym.make("CartPole-v1")

print("=" * 50)
print("CartPole-v1 environment info")
print("=" * 50)
print(f"  Observation space:  {env.observation_space}")
print(f"  Action space:  {env.action_space}")
print(f"  Observation upper bound:  {env.observation_space.high}")
print(f"  Observation lower bound:  {env.observation_space.low}")
print(f"  Termination condition:  position > ±{env.unwrapped.x_threshold}, "
      f"angle > ±{env.unwrapped.theta_threshold_radians:.4f} rad "
      f"(≈ ±{np.degrees(env.unwrapped.theta_threshold_radians):.0f}°)")
print("=" * 50)

CartPole-v1 environment info
  Observation space:  Box([-4.8               -inf -0.41887903        -inf], [4.8               inf 0.41887903        inf], (4,), float32)
  Action space:  Discrete(2)
  Observation upper bound:  [4.8               inf 0.41887903        inf]
  Observation lower bound:  [-4.8               -inf -0.41887903        -inf]
  Termination condition:  position > ±2.4, angle > ±0.2094 rad (≈ ±12°)


## 3. 构建 PPO 模型

`MlpPolicy` = 一个小型全连接网络，同时输出策略（选哪个动作）和价值估计。
小网络在 MPS 上不一定比 CPU 快，想对比就把 `device` 改成 `"cpu"` 重跑上面的 cell。

In [3]:
model = PPO("MlpPolicy", env, verbose=1, device=device)

if device == "mps":
    print(
        "Note: SB3 MlpPolicy on Apple MPS works, but CPU is often faster for "
        "small networks. Try device = 'cpu' to compare."
    )
model.policy

Using mps device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Note: SB3 MlpPolicy on Apple MPS works, but CPU is often faster for small networks. Try device = 'cpu' to compare.


/Users/pengqianhan/Documents/GitHub/hands-on-modern-rl-pq/.venv/lib/python3.10/site-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


ActorCriticPolicy(
  (features_extractor): FlattenExtractor(
    (flatten): Flatten(start_dim=1, end_dim=-1)
  )
  (pi_features_extractor): FlattenExtractor(
    (flatten): Flatten(start_dim=1, end_dim=-1)
  )
  (vf_features_extractor): FlattenExtractor(
    (flatten): Flatten(start_dim=1, end_dim=-1)
  )
  (mlp_extractor): MlpExtractor(
    (policy_net): Sequential(
      (0): Linear(in_features=4, out_features=64, bias=True)
      (1): Tanh()
      (2): Linear(in_features=64, out_features=64, bias=True)
      (3): Tanh()
    )
    (value_net): Sequential(
      (0): Linear(in_features=4, out_features=64, bias=True)
      (1): Tanh()
      (2): Linear(in_features=64, out_features=64, bias=True)
      (3): Tanh()
    )
  )
  (action_net): Linear(in_features=64, out_features=2, bias=True)
  (value_net): Linear(in_features=64, out_features=1, bias=True)
)

## 4. 训练

`verbose=1` 会滚动打印一张表格，重点看 `rollout/ep_rew_mean`（平均回合回报）是否上升。
CartPole-v1 的满分是 500。

先用一个小的 `total_timesteps` 跑通流程（几秒钟），确认没问题后再改成 80000 正式训练。

In [4]:
TOTAL_TIMESTEPS = 20000   # 想复现脚本的效果就改成 80000

model.learn(total_timesteps=TOTAL_TIMESTEPS)

---------------------------------
| rollout/           |          |
|    ep_len_mean     | 24.5     |
|    ep_rew_mean     | 24.5     |
| time/              |          |
|    fps             | 189      |
|    iterations      | 1        |
|    time_elapsed    | 10       |
|    total_timesteps | 2048     |
---------------------------------
----------------------------------------
| rollout/                |            |
|    ep_len_mean          | 24.5       |
|    ep_rew_mean          | 24.5       |
| time/                   |            |
|    fps                  | 177        |
|    iterations           | 2          |
|    time_elapsed         | 23         |
|    total_timesteps      | 4096       |
| train/                  |            |
|    approx_kl            | 0.00809003 |
|    clip_fraction        | 0.0935     |
|    clip_range           | 0.2        |
|    entropy_loss         | -0.686     |
|    explained_variance   | -0.00961   |
|    learning_rate        | 0.0003     |
|   

## 5. 评估并保存

`evaluate_policy` 用确定性策略跑 10 个回合，返回平均回报和标准差。

In [ ]:
mean_reward, std_reward = evaluate_policy(model, env, n_eval_episodes=10)
print(f"Training complete! Mean reward: {mean_reward} +/- {std_reward}")

model.save("output/ppo_cartpole")
env.close()

## 6. 演示：把学到的策略跑起来

`render_mode=None` 时不弹窗，只打印分数（notebook 里推荐这样）。
想看小车动画，把 `GUI = True`，会弹出一个独立窗口；
在远程/无显示器环境下弹窗会失败，保持 `False` 即可。

In [ ]:
GUI = False

vis_env = gym.make("CartPole-v1", render_mode="human" if GUI else None)
model = PPO.load("output/ppo_cartpole")

for episode in range(5):
    obs, info = vis_env.reset()
    done, truncated, score = False, False, 0
    while not (done or truncated):
        action, _states = model.predict(obs, deterministic=True)
        obs, reward, done, truncated, info = vis_env.step(action)
        score += reward
    print(f"  Episode {episode + 1} score: {score}")

vis_env.close()

## 7. 看清楚一步 step 里发生了什么

上面的循环把细节藏起来了。这个 cell 单步展开前 5 步，
观察 observation / action / reward / done 的具体取值。

In [ ]:
demo_env = gym.make("CartPole-v1")
obs, info = demo_env.reset(seed=0)
print("初始观测:", obs)

for t in range(5):
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, done, truncated, info = demo_env.step(action)
    print(f"step {t}: action={action}  obs={np.round(obs, 3)}  reward={reward}  done={done}")

demo_env.close()